# Experiment 2 eye features

Use a CPU runtime and run the cells in order. The notebook reads the verified ZIP from My Drive / MML_BBBD. It uses attentive recordings and the calibration assumption documented in Proposed D-32 and DV-03. Outputs contain eye features and exclusion flags only.


In [ ]:
import hashlib
import shutil
import subprocess
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/MML_BBBD")
archive = root / "experiment2.zip"
assert archive.is_file(), f"Archive missing: {archive}"
assert archive.stat().st_size == 5_601_886_507, "Unexpected ZIP size"
with archive.open("rb") as stream:
    digest = hashlib.file_digest(stream, "md5").hexdigest()
assert digest == "e9671316f97e1c3ec1d86a03e0a1ab28", "ZIP checksum mismatch"
print("Archive checksum verified")

repo = Path(tempfile.mkdtemp(prefix="bbbd_exp2_"))
def run(*args):
    subprocess.run(args, cwd=repo, check=True)

run("git", "clone", "https://github.com/pbhatt0022/bbbd_mml.git", str(repo))
run("git", "checkout", "--detach", "8d8e97968fc84586f4ca42010b7b1721695a2428")


## Install and test

The embedded files are the local extractor, test, decision and deviation logs, and registry through R-013. This overlay will be recorded as a dirty checkout. The backup retains the exact source and configuration. Run this notebook once for this registry snapshot.


In [ ]:
bundled = {'src/bbbd/exp2_eye_features.py': '"""Generate registered attentive Experiment 2 eye features directly from the raw ZIP."""\nimport argparse\nimport gzip\nimport json\nimport re\nfrom zipfile import ZipFile\n\nimport pandas as pd\n\nfrom bbbd import eye, features, runlog\nfrom bbbd.config import load_config\n\nGAZE = re.compile(\n    r"sub-(\\d+)/ses-01/eyetrack/sub-\\1_ses-01_task-stim(\\d+)_gaze_visualangle_eyetrack\\.tsv\\.gz"\n)\n\n\ndef read_eye(archive, name, columns):\n    sidecar = json.loads(archive.read(name.removesuffix(".tsv.gz") + ".json"))\n    if (sidecar["Columns"] != columns or sidecar["SamplingFrequency"] != 128\n            or sidecar["StartTime"] != 0\n            or not sidecar["TaskName"].endswith(", Attentive Condition")):\n        raise ValueError(f"Unexpected metadata: {name}")\n    if columns[0] == "x" and any(sidecar[c]["Units"] != "screen pixels" for c in ("x", "y")):\n        raise ValueError(f"Unexpected gaze units: {name}")\n    with archive.open(name) as member, gzip.GzipFile(fileobj=member) as raw:\n        data = pd.read_csv(raw, sep="\\t", header=None, dtype=float)\n    if data.empty or data.shape[1] != len(columns):\n        raise ValueError(f"Unexpected table shape: {name}")\n    data.columns = columns\n    return data\n\n\ndef extract(archive, cfg):\n    screen = cfg["eye"]["screen"]\n    scale = screen["px_per_deg"] or eye.px_per_deg_from_screen(\n        screen["width_px"], screen["width_cm"], screen["distance_cm"]\n    )\n    rows = []\n    keys = set()\n    for name in sorted(archive.namelist()):\n        match = GAZE.fullmatch(name)\n        if not match:\n            continue\n        subject, video = match.groups()\n        key = (int(subject), int(video))\n        if key in keys:\n            raise ValueError(f"Duplicate recording key: {key}")\n        keys.add(key)\n        gaze = read_eye(archive, name, ["x", "y", "vdx", "vdy"])\n        pupil = read_eye(archive, name.replace("_gaze_visualangle_", "_pupil_"), ["pupil_size"])\n        if len(gaze) != len(pupil):\n            raise ValueError(f"Unequal gaze and pupil lengths: {name}")\n        row = features.eye_features(gaze.x.to_numpy()/scale, gaze.y.to_numpy()/scale,\n                                    pupil.pupil_size.to_numpy(), 128, cfg["eye"])\n        rows.append(dict(experiment=2, subject=f"sub-{subject}", video=int(video),\n                         session="ses-01", variant="V1",\n                         excluded=row["eye_invalid_frac"] > cfg["exclusions"]["eye_max_invalid_frac"],\n                         **row))\n    if not rows:\n        raise ValueError("No matching attentive Experiment 2 recordings")\n    return pd.DataFrame(rows), scale\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("archive")\n    args = parser.parse_args()\n    cfg = load_config()\n    with ZipFile(args.archive) as archive:\n        table, scale = extract(archive, cfg)\n    run = runlog.start_run(cfg, "eye_features_exp2", "exp2_eye_features", member="A")\n    runlog.save_table(run, table, "eye_features_exp2.csv")\n    runlog.save_json(run, {"archive": "experiment2.zip", "px_per_deg": scale,\n                          "calibration": "D-26 assumption applied under Proposed D-32",\n                          "session": "ses-01", "n_rows": len(table),\n                          "n_excluded": int(table.excluded.sum())}, "eye_features_exp2.json")\n    runlog.finish_run(run, note="D-32/DV-03 Experiment 2 eye features; preprocessing only.")\n\n\nif __name__ == "__main__":\n    main()\n', 'tests/test_exp2_eye_features.py': 'import gzip\nimport io\nimport json\nfrom zipfile import ZipFile\n\nimport pytest\n\nfrom bbbd.exp2_eye_features import read_eye\n\n\ndef test_eye_reader_checks_author_condition():\n    name = "gaze.tsv.gz"\n    metadata = {"Columns": ["x", "y"], "SamplingFrequency": 128, "StartTime": 0,\n                "x": {"Units": "screen pixels"}, "y": {"Units": "screen pixels"}}\n    for condition in ("Attentive", "Distracted"):\n        with ZipFile(io.BytesIO(), "w") as archive:\n            archive.writestr(name, gzip.compress(b"1\\t2\\n"))\n            archive.writestr("gaze.json", json.dumps(\n                dict(metadata, TaskName=f"Stim 01, {condition} Condition")\n            ))\n            if condition == "Attentive":\n                assert read_eye(archive, name, ["x", "y"]).iloc[0].tolist() == [1, 2]\n            else:\n                with pytest.raises(ValueError, match="metadata"):\n                    read_eye(archive, name, ["x", "y"])\n', 'docs/decision_log.md': '# Decision Log\n\nEvery choice that could change a result is logged here **before** it is used. Status moves Open → Proposed → Decided. If a decision changes later, mark the old row **Revisited** and add a new row explaining why; never overwrite. "Proposed" rows are recommendations and are not decided until both members agree (initial both in the Agreed column).\n\nCommit each change with a message naming the row, e.g. `D-17: decided`.\n\nMigrated from the Project Hub (Claude Docs) on 6 Nov 2026. From now on this file is the master copy.\n\n| ID | Status | Decide by | Question | Options considered | Current answer and why | Source | Agreed |\n| --- | --- | --- | --- | --- | --- | --- | --- |\n| D-00 | Decided | | Which dataset? | BBBD; other education datasets | BBBD: 5 experiments, synchronised EEG/ECG/eye/respiration + quizzes, CC BY 4.0 | Project proposal; [Madsen 2026](https://www.nature.com/articles/s41597-026-07215-1) | A, B |\n| D-01 | Decided | | Where does preprocessing start? | Authors\' derivatives; raw files | Raw BDF (EEG/ECG/EOG/resp) and raw TSV (eye). Authors\' derivatives used only as a validation reference. **Confirmed by the professor on 15 Oct** ("do all preprocessing on our own") | Team, 12 Oct; professor 15 Oct | A, B |\n| D-02 | Decided | | Compute and storage | Local; Colab Pro | Colab Pro; data on shared Google Drive; code in GitHub (D-19) | Team, 12 Oct | A, B |\n| D-03 | Proposed | 6 Nov | Which experiments are in scope? | All 5; EEG ones only (2–5); a subset | Eye on Exps 1–5, EEG/ECG on 2–5, respiration on 4–5; Exp 5 second (incentive) session analysed separately | Methods plan §3 | |\n| D-04 | Proposed | 6 Nov | Prediction targets | Quiz score; attention condition; engagement; prior knowledge | Quiz score primary; attend vs distract auxiliary; engagement exploratory only | Literature Review §3 | |\n| D-05 | Decided (scope) / Proposed (design) | 6 Nov | Must the project use deep learning? | Classical only; classical + DL component; DL only | **Professor 15 Oct: include some deep learning.** Design: one substantial DL track (RQ4, D-17) alongside classical confirmatory models, because N ≈ 600 learner × video rows is small for end-to-end DL | Professor 15 Oct; [Bodonhelyi 2026](https://arxiv.org/abs/2604.09569) | |\n| D-06 | Decided | | Download the YouTube videos (S2)? | Yes; no | No: S2 dropped (D-18) | Methods plan §2 | |\n| D-07 | Proposed | 6 Nov | Where is the preregistration timestamped? | PDF on Drive; OSF; git | `docs/preregistration.md` frozen with git tag `prereg-v1` pushed to GitHub (timestamped by GitHub); PDF copy on Drive | Preregistration | |\n| D-08 | Decided | | Final dates | — | Submission Mon 16 Nov 2026 EOD (PPT, code with outputs, report); presentation Wed 18 Nov 2026 | Final Project Instructions | A, B |\n| D-09 | Proposed | 6 Nov | Unit of analysis and grouping | Learner × video rows grouped by learner; learner averages | Learner × video rows; folds grouped by learner, so no learner is in both train and test | [Bodonhelyi 2026](https://arxiv.org/abs/2604.09569) on user-dependent inflation | |\n| D-10 | Proposed | 6 Nov | Video difficulty in the target | Raw proportion; z-score within video; video covariate | z-score within video and experiment, so models explain differences between learners | Methods plan §3 | |\n| D-11 | Proposed | 6 Nov | ISC reference group | All other learners; training-fold only | Training-fold learners only, recomputed per fold; training learners use leave-one-out within the training fold (`bbbd.isc`) | Methods plan §5 | |\n| D-12 | Proposed | 6 Nov | EEG eye-artifact removal (default) | EOG regression; ICA; none | EOG regression (authors\' approach) as V1; ICA as the RQ3 variant V3 | [Madsen 2026](https://www.nature.com/articles/s41597-026-07215-1); [Delorme 2023](https://pmc.ncbi.nlm.nih.gov/articles/PMC9911389) | |\n| D-13 | Proposed | 6 Nov | 16 Hz noise in Exps 4–5 | Notch 16 Hz + harmonics; drop channels; exclude | Detect per recording from the spectrum (`eeg.has_peak`, ratio > 3 vs flanks); notch if present; log which recordings | Madsen 2026 known issues | |\n| D-14 | Proposed | 6 Nov | Tooling | MNE + NeuroKit2 + scikit-learn; MATLAB | Python only: MNE, NeuroKit2, pandas/NumPy, scikit-learn, PyTorch | [NeuroKit2](https://github.com/neuropsychology/NeuroKit) | |\n| D-15 | Proposed | 6 Nov | Primary metric | Pearson; Spearman; R²; MAE | Spearman ρ primary; R² vs mean baseline and MAE secondary | Preregistration | |\n| D-16 | Proposed | 6 Nov | Split of work | By modality; by stage | A = eye + targets + models + DL fusion; B = EEG + ECG + resp + DL encoders/contrastive | Methods plan §8 | |\n| D-17 | Proposed | 6 Nov | Deep learning design | End-to-end on quiz score; pretrain on window tasks then reuse; none | Encoders (EEGNet-style for EEG, 1D-CNN for eye/cardiac) trained on 4-s windows for (A) contrastive EEG↔eye alignment and (B) attend vs distract with early / late / cross-attention fusion and modality dropout; (C) pooled embeddings → ridge for quiz score. Encoders retrained in each outer fold | Methods plan §6b; [Lawhern 2018](https://doi.org/10.1088/1741-2552/aace8c) | |\n| D-18 | Proposed | 6 Nov | Scope cuts for 10 days | Keep all; cut | RQ3 reduced to V1 vs V3 (ICA); V2 dropped. S1 (PID) optional on Thu 12 Nov. S2 (question-level) dropped to future work. Drop order if late: S1 → RQ3 → leave-one-video-out | Methods plan §8 | |\n| D-19 | Proposed | 6 Nov | Version control and source of truth | Drive only; GitHub + Drive | Private GitHub repo `pbhatt0022/bbbd_mml` for code and docs; data on Drive only. Repo Markdown is the master; the Hub is a reading view | Methods plan §1 | |\n| D-20 | Proposed | 6 Nov | Order of average reference and EOG regression | Reference before; reference after | Reference **before** EOG regression: MNE\'s `EOGRegression` refuses to run without a reference, and MNE\'s tutorial sets it first. Found while testing `bbbd.eeg` on 6 Nov | MNE-Python `EOGRegression` | |\n| D-21 | Proposed | 6 Nov | DL window length | 2 s; 4 s; 8 s | 4 s, 50% overlap: long enough for alpha power and several saccades, short enough for thousands of windows per experiment. 8 s for the cardiac encoder if 4 s holds too few beats | Methods plan §6b | |\n| D-22 | Proposed | 7 Nov | Python and core package versions | System default 3.14; 3.12; 3.11 | **Python 3.12** in `.venv`: torch has no 3.14 wheels, and 3.14 is the machine\'s `py` default, so the version must be pinned or each member gets a different one. Resolved on 6 Nov: numpy 2.5.4, scipy 1.18.1, pandas 3.1.1, scikit-learn 1.9.1, torch 2.15.0, mne 1.13.2, neurokit2 0.3.0, pyarrow 26.0.1. `pyproject.toml` sets only lower bounds, so these are what resolved today, not a guarantee; Colab may differ, which matters for RQ0 beat matching and any seeded split | Repo setup 6 Nov; `pyproject.toml` | |\n| D-23 | Proposed | 7 Nov | Repository visibility | Private; public | **Public.** Chosen by A on 6 Nov 2026 after the repo was found to be public (`"private": false`), which contradicts the wording of D-19. No dataset files are in the repo, so nothing under the Zenodo CC BY 4.0 terms is exposed; `docs/` (preregistration, decision log, methods plan) is world-readable, which also makes the D-07 `prereg-v1` timestamp publicly checkable. **D-19\'s "Private GitHub repo" wording is superseded by this row**; mark D-19 Revisited once both members agree | Repo check 6 Nov; supersedes D-19 | |\n| D-24 | Proposed | 9 Nov | How to get gaze into degrees for saccade detection | Sidecar `vdx`/`vdy` as-is; screen geometry + `px_to_deg`; estimate px/deg from the data | **Estimate px/deg** (`eye.estimate_px_per_deg`), ~24.9 on Exp 1, until the real screen dimensions are found. Exp 1 `x`/`y` are screen pixels (measured in R-001: x -377..1259, y -1010..1924) despite `visualangle` in the filename, and `vdx`/`vdy` are smoothed (p99.9 = 26 deg/s, under the 30 deg/s threshold), so thresholding them gave 2.7 saccades/min and 82-s fixations in R-001. `config.yaml eye.screen.px_per_deg` overrides the estimate once geometry is known | R-001 diagnostics, 8 Nov | |\n| D-25 | Proposed | 9 Nov | Blink source for Exp 1 | Gaze track-loss; pupil dropout | **Pupil dropout**, per methods plan §4. Exp 1 does ship a `pupil_size` file per recording (263 `*_pupil_eyetrack.tsv.gz`, paired on sub/ses/task), which the Dataset Inventory had listed as an open question. R-001 used track-loss and found 0 blinks, because `invalid_frac` is 0.00 in every gaze file | R-001 diagnostics, 8 Nov | |\n| D-26 | Proposed | 9 Nov | Pixels per degree for Exp 1 gaze | Per-recording estimate (D-24); screen geometry; authors\' derivative in degrees | **Screen geometry: 33.64 px/deg.** The BBBD preprint states "a 27" monitor approximately 60 cm from the subject" (bioRxiv 10.1101/2025.04.29.651259, Procedure), giving 59.77 cm width. **The resolution is stated in no source we could read**; 1920x1080 is *inferred* because it yields 33.64 px/deg against the independent data estimate of 34.31 (implying 1958 px), while 2560x1440 would give 44.85 and is ruled out. **Supersedes the estimator in D-24**, which scattered 7.6-117.6 across recordings (relative IQR 0.69) and is kept only as a cross-check. Caveat: the authors convert with the EyeLink\'s per-sample resolution columns, which vary with gaze eccentricity and are absent from the released files, so any constant is an approximation | Preprint Procedure; R-002 calibration | |\n| D-27 | Proposed | 10 Nov | Which quiz questions make up the target | Memory items only; memory + domain; separate models per type | **Memory items only.** They test what the learner took from the video, which is what RQ1 predicts. `domain` questions probe prior knowledge of the subject, so including them would mix learning with what the learner already knew; the preregistration (§3) lists prior knowledge as an exploratory *covariate*, never as the outcome. Exp 1 has 1,829 memory and 1,457 domain item rows, and the inventory\'s per-video means (0.64/0.56/0.60/0.60/0.79) are memory items, so `05_targets` checks that this filter reproduces them | Preregistration §3; `05_targets` | |\n| D-28 | Proposed | 10 Nov | What leave-one-video-out holds out | Video only; video and learner together; drop LOVO | **Video only**, as the methods plan\'s second scheme intends. The same learners are therefore in train and test for different videos, so a model can lean on learner-specific quirks and LOVO numbers will usually look better than the learner-grouped ones. They answer a different question (generalise to a **new video**, not a **new learner**) and are **not comparable** to the primary 5-fold grouped results. Every reported number must name its scheme. Found when the leakage check in `06_folds` flagged all 27 learners across LOVO folds | `06_folds`; methods plan §7 | |\n| D-29 | Proposed | 10 Nov | Which Exp 1 session is attend | ses-01; ses-02; wait for documentation | **ses-01 = attend, ses-02 = distract.** Two independent strands: R-004 blink rates reproduce the 14 Oct attend→distract figures exactly (1.25 → 2.90 per min, t = 3.04, p = .005), and blinks come from pupil so they do not depend on the px/deg calibration; and the preprint states Exp 1 ran the attentive session first ("After they watched all the videos, they watched them again in a distracted condition"). **No source states the label mapping directly.** Everything in RQ1/RQ2 is computed on the attend session alone (prereg §2), so if this is backwards every such number is on the wrong session | R-004; bioRxiv 10.1101/2025.04.29.651259 Procedure | |\n| D-30 | Proposed | 11 Nov | Which signal the eye exclusion rule is measured on | Gaze validity; pupil validity; both | **Pupil.** The released Exp 1 gaze is already interpolated and reads `invalid_frac` = 0.00 in all 263 recordings, so a gaze-based rule would exclude nothing and the preregistered >50% threshold (§2) would be dead text. Pupil has real dropout: R-008 found 15 of 135 recordings above 50%, up to 99.9% (sub-31 video 5, which yields zero saccades). Applying it drops **sub-22 and sub-31 entirely**, leaving 25 learners and 120 rows, so `results/folds.json` must be rebuilt. This is the preregistered rule being *implemented*, not changed; only the choice of signal is new | Prereg §2; R-008 | |\n\n| D-31 | Proposed | 13 Nov | Reconcile R-012 implementation with the frozen analysis plan | Retain the documented implementation only; transparently rerun ridge with matched inference and nested tuning | Preserve R-012 unchanged. Run a separately registered, post-result correction using R-006 targets, R-010 features, the exact R-011 outer folds and the same eight predictors. Use three-fold GroupKFold within each outer training set, tune the existing 30 log-spaced ridge alphas from 1e-3 to 1e4 by mean inner-fold MSE, and fit scaling inside each inner training set. Average predictions over all ten repeats for BOTH the observed statistic and every within-video permutation. Keep 1,000 permutations, 2,000 learner bootstrap resamples, seed 2026, exclusions and thresholds unchanged. Report conventional R² and separately named skill against averaged outer-training means. Report raw eye p only: across-modality Holm remains pending, with no automatic H1a support claim. This is a correction prepared after seeing R-012, not a new blind preregistration. Settings live in config.yaml; joint agreement remains outstanding. | Static inspection of notebook 60 and cv.py; prereg §4; DV-02 | |\n\n| D-32 | Proposed | 14 Nov | Experiment 2 eye processing and prediction inputs | Reuse V1 eye features with explicit calibration assumption; defer modeling | Use archive sidecar condition labels, attentive recordings only, exact gaze/pupil pairing and equal sample counts. Reuse V1 settings including padded pupil invalid fraction. Use D-26 screen scale as an explicit assumption, supported by shared apparatus but unverified resolution. Use memory proportion from the verified Experiment 2 phenotype. Recovered physiology tables remain diagnostic. Resolve ECG eligibility and formal provenance before freezing a separate common-sample fold file. | Verified archive metadata and recovery audit; DV-03 | |\n\n## Professor feedback (15 Oct 2026)\n\n| Feedback point | Decision affected | Action |\n| --- | --- | --- |\n| The progress presentation was received well; keep most of the plan | All | Final plan builds on the 15 Oct deck |\n| Do all preprocessing on our own | D-01 | Confirmed; raw files only, authors\' derivatives for validation only |\n| Use some deep learning | D-05, D-17 | Added RQ4 deep learning track |\n| Replicating results from the original paper is fine | RQ0 | RQ0 kept as the replication part |\n', 'docs/deviations.md': '# Deviations from the Preregistration\n\nAfter the git tag `prereg-v1`, any change to hypotheses, exclusions, variables, metrics, splits or models is logged here before results that depend on it are reported. Each row says what changed, why, and the expected effect on results. Report the preregistered analysis as well when it can still be run.\n\n| # | Date | Section changed | What changed | Why | Expected effect on results | Decision Log row |\n| --- | --- | --- | --- | --- | --- | --- |\n| DV-01 | 6 Nov 2026 | Freeze date | Planned freeze moved from 18 Oct to 6 Nov 2026 | Repo set up late; no feature had been correlated with quiz score (confirmed by both members, 9 Nov 2026) | None; the confirmation held | D-07 |\n| DV-02 | 13 Nov 2026 | §4.1–4.4 implementation and R-012 reporting | Retrospective disclosure: R-012 tuned with default RidgeCV rather than the stated inner three folds; its permutation used only the first repeat while its estimate/CI used ten-repeat averaged predictions; its R² used the evaluated-target mean, not training-fold mean predictions. Its Holm family contained only eye. Prospective correction: separately register nested learner-grouped three-fold ridge tuning with inner-fold scaling and mean MSE selection, refit all ten repeats for each permutation, retain conventional R² and separately report training-mean skill, and leave across-modality inference pending. | Discovered by code inspection after R-012 was seen. D-31 specifies previously underspecified inner grouping/scoring and the matched statistic. This row is recorded before the new run; it does not retroactively make R-012 compliant. R-012 and the tagged files are retained. The new run is labelled post-result correction under DV-02, not unqualified confirmatory. | Predictions, CI and p may change in either direction; support may disappear. No change is contingent on achieving significance. All thresholds, targets, features, outer folds and resampling counts stay fixed. | D-31 (Proposed; humans have not initialled) |\n\n| DV-03 | 14 Nov 2026 | Experiment 2 extension, §2–4 | Prepare a separate attentive-session Experiment 2 cohort and report the assumed D-26 eye calibration. Any later common-sample split artifact will be separate from frozen Experiment 1 folds and specified before fitting. | Experiment 1 folds contain only Experiment 1 keys. Experiment 2 resolution and ECG eligibility are unresolved. | Cohort composition and absolute angle features may differ. No change to numerical thresholds. | D-32 (Proposed) |\n', 'docs/run_registry.csv': 'run_id,started_utc,finished_utc,name,notebook,member,git_hash,git_dirty,config_hash,seed,output_dir,note\nR-001,2026-11-07T09:38:28+00:00,2026-11-07T09:48:45+00:00,eye_rq0_exp1,10_eye_preproc,A,e831839,False,ac8b369b85,2026,results/R-001_eye_rq0_exp1,"RQ0 eye Exp 1: attend vs distract event rates, V1. No quiz scores."\nR-002,2026-11-08T11:29:25+00:00,2026-11-08T11:35:29+00:00,eye_rq0_exp1,10_eye_preproc,A,bedf2a0,False,47c0d2645a,2026,results/R-002_eye_rq0_exp1,"RQ0 eye Exp 1: attend vs distract event rates, V1. No quiz scores."\nR-003,2026-11-09T07:29:42+00:00,2026-11-09T07:36:57+00:00,eye_rq0_exp1,10_eye_preproc,A,0ad7b99,False,1f606e1458,2026,results/R-003_eye_rq0_exp1,"RQ0 eye Exp 1: attend vs distract event rates, V1. No quiz scores."\nR-004,2026-11-09T07:37:38+00:00,2026-11-09T07:38:13+00:00,eye_rq0_exp1,10_eye_preproc,A,01405b7,False,1f606e1458,2026,results/R-004_eye_rq0_exp1,"RQ0 eye Exp 1: attend vs distract event rates, V1. No quiz scores."\nR-005,2026-11-09T08:09:43+00:00,2026-11-09T08:09:43+00:00,targets_exp1,05_targets,A,0f42319,False,1f606e1458,2026,results/R-005_targets_exp1,"Targets Exp 1: memory-item proportion correct, z within video. No features involved."\nR-006,2026-11-09T08:17:11+00:00,2026-11-09T08:17:11+00:00,targets_exp1,05_targets,A,e69d2ab,False,1f606e1458,2026,results/R-006_targets_exp1,"Targets Exp 1: memory-item proportion correct, z within video. No features involved."\nR-007,2026-11-09T08:26:14+00:00,2026-11-09T08:26:14+00:00,folds_exp1,06_folds,A,dc6f816,False,1f606e1458,2026,results/R-007_folds_exp1,"Folds Exp 1: 5-fold grouped by learner x10 repeats, plus LOVO. No target values used."\nR-008,2026-11-09T10:13:43+00:00,2026-11-09T10:13:43+00:00,eye_features_exp1,50_features_individual,A,4332e40,False,1f606e1458,2026,results/R-008_eye_features_exp1,"Individual eye features, Exp 1 attend session, V1. No target column, no correlation with score."\nR-009,2026-11-09T10:19:14+00:00,2026-11-09T10:19:15+00:00,eye_features_exp1,50_features_individual,A,1df1815,False,1f606e1458,2026,results/R-009_eye_features_exp1,"Individual eye features, Exp 1 attend session, V1. No target column, no correlation with score."\nR-010,2026-11-09T10:21:49+00:00,2026-11-09T10:21:49+00:00,eye_features_exp1,50_features_individual,A,487c6d0,False,1f606e1458,2026,results/R-010_eye_features_exp1,"Individual eye features, Exp 1 attend session, V1. No target column, no correlation with score."\nR-011,2026-11-09T10:30:07+00:00,2026-11-09T10:30:07+00:00,folds_exp1,06_folds,A,a4eaf96,False,1f606e1458,2026,results/R-011_folds_exp1,"Folds Exp 1: 5-fold grouped by learner x10 repeats, plus LOVO. No target values used."\nR-012,2026-11-13T11:27:48+00:00,2026-11-13T11:27:48+00:00,classical_eye_exp1,60_classical_models,A,45d3d34,False,1f606e1458,2026,results/R-012_classical_eye_exp1,"RQ1/RQ2 eye-only, Exp 1, individual features. Confirmatory."\nR-013,2026-11-13T12:07:42+00:00,2026-11-13T16:51:32+00:00,eye_reconciliation,85_statistics,A,8d8e979,False,f81d91f284,2026,results/R-013_eye_reconciliation,D-31/DV-02 post-result correction COMPLETE; Holm pending; original R-012 preserved\n'}

for name, content in bundled.items():
    (repo / name).write_text(content, encoding="utf-8")
run(sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]")
run(sys.executable, "-m", "pytest", "-q", "tests/test_exp2_eye_features.py")


## Extract and save

Run after the test passes. The final cell saves the registered feature table and run metadata to a new folder in 04_results. Keep the executed notebook with its outputs.


In [ ]:
existing = list((repo / "results").glob("R-*_eye_features_exp2"))
if not existing:
    run(sys.executable, "-m", "bbbd.exp2_eye_features", str(archive))
output, = (repo / "results").glob("R-*_eye_features_exp2")
summary = output / "eye_features_exp2.json"
assert summary.is_file(), "Run is incomplete. Inspect the error before restarting."
registry = repo / "docs/run_registry.csv"
assert f"{output.name.split('_')[0]}," in registry.read_text(), "Run is missing from registry"

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
backup = root / "04_results" / f"exp2_eye_{stamp}"
shutil.copytree(output, backup / output.name)
for name in bundled:
    target = backup / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(repo / name, target)
shutil.copy2(repo / "config.yaml", backup / "config.yaml")
print("Saved to:", backup)
print(summary.read_text())
